In [11]:
import datetime                             # manejo de fechas (timestamps)
import librosa                              # procesamiento de audio
import numpy as np                          # operaciones numéricas
import pandas as pd                         # manejo de datos tabulares
import math                                 # funciones matemáticas
import matplotlib.pyplot as plt             # visualización
from pathlib import Path                    # gestión de rutas
from pydub import AudioSegment              # conversión de formatos (m4a → wav)
from IPython.display import Audio, display  # reproducción en notebook
import soundfile as sf                      # lectura/escritura de audio
import noisereduce as nr                    # reducción de ruido
import random                               # generación de números aleatorios
from scipy.signal import butter, filtfilt   # filtros de señal
from silero_vad import load_silero_vad, get_speech_timestamps      # modelo de VAD preentrenado
import torch

In [ ]:
# Extrae metadatos básicos y define flags necesarios para el pipeline ASR
# Asume configuración fija: audio final en WAV, mono, 16 kHz
def prepare_audio_metadata(
    audio_dataset: list[dict]
) -> list[dict]:
    
    TARGET_SR = 16000  # Requisito estándar para ASR
    
    for item in audio_dataset:

        audio_path = item["path"]

        try:
            # Lectura eficiente de metadatos
            info = sf.info(audio_path)

            sample_rate = info.samplerate
            num_channels = info.channels
            duration_sec = info.duration
            subtype = info.subtype

        except Exception:
            # Fallback para formatos no soportados directamente
            audio = AudioSegment.from_file(audio_path)

            sample_rate = audio.frame_rate
            num_channels = audio.channels
            duration_sec = len(audio) / 1000
            subtype = "compressed"

        # Metadatos
        item["sample_rate"] = sample_rate
        item["num_channels"] = num_channels
        item["duration_sec"] = duration_sec
        item["subtype"] = subtype

        # Flags de procesamiento (decisiones fijas para ASR)
        item["needs_resample"] = sample_rate != TARGET_SR
        item["needs_mono"] = num_channels > 1
        item["needs_conversion"] = audio_path.suffix.lower() != ".wav"

        # Clasificación básica de calidad
        if duration_sec < 1:
            quality_flag = "too_short"
        elif duration_sec > 60:
            quality_flag = "too_long"
        else:
            quality_flag = "ok"

        item["quality_flag"] = quality_flag

    return audio_dataset


# Estandariza los audios al formato requerido por ASR (WAV, mono, 16 kHz)
# Guarda los audios en disco y añade la ruta estandarizada al dataset
def standardize_audio_dataset(
    audio_dataset: list[dict],
    output_dir: Path
) -> tuple[list[dict], int, int, list[tuple[str, str]]]:
    
    TARGET_SR = 16000  # Frecuencia fija para ASR
    
    standardized_count = 0
    error_count = 0
    error_files = []
    
    for item in audio_dataset:

        input_path = item["path"]
        output_path = output_dir / f"{input_path.stem}.wav"

        try:
            # Conversión directa para formatos comprimidos
            if item["needs_conversion"]:

                audio = AudioSegment.from_file(input_path)
                audio = audio.set_channels(1).set_frame_rate(TARGET_SR)
                audio.export(output_path, format="wav")

            else:
                # Carga del audio original
                y, sr = librosa.load(input_path, sr=None, mono=False)

                # Conversión a mono
                if item["needs_mono"]:
                    y = librosa.to_mono(y)

                # Remuestreo a 16 kHz
                if item["needs_resample"]:
                    y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
                    sr = TARGET_SR

                # Guardado del audio estandarizado
                sf.write(output_path, y, sr)

            # Guardar ruta del audio estandarizado
            item["standardized_path"] = output_path
            standardized_count += 1

        except Exception as e:
            error_count += 1
            error_files.append((input_path.name, str(e)))

    return audio_dataset



# Carga los audios estandarizados en memoria para su procesamiento posterior
# Devuelve una lista con la señal y frecuencia de muestreo de cada audio
def load_standardized_audio(
    audio_dataset: list[dict]
) -> list[dict]:
    
    baseline_dataset = []
    
    for item in audio_dataset:

        try:
            # Carga del audio estandarizado
            y, sr = librosa.load(item["standardized_path"], sr=None)

            baseline_dataset.append({
                "audio_id": item.get("audio_id", item["path"].stem),
                "y": y,
                "sr": sr
            })

        except Exception as e:
            print(f"Error cargando {item['standardized_path'].name}: {e}")

    return baseline_dataset



# Aplica eliminación de silencios en los extremos (trim) y padding para preservar contexto
def apply_trim_and_padding(
    audio_dataset: list[dict],
    top_db: int = 20,
    padding_ms: int = 300
) -> list[dict]:
    
    TARGET_SR = 16000  # Frecuencia fija del pipeline
    padding_samples = int((padding_ms / 1000) * TARGET_SR)
    
    output_dataset = []
    
    for item in audio_dataset:

        y = item["y"]
        sr = item["sr"]

        # Eliminación de silencios en extremos
        y_trim, _ = librosa.effects.trim(y, top_db=top_db)

        # Generación de padding
        pad = np.zeros(padding_samples)

        # Aplicación de padding
        y_final = np.concatenate([pad, y_trim, pad])

        output_dataset.append({
            "audio_id": item["audio_id"],
            "y": y_final,
            "sr": sr
        })

    return output_dataset


# Aplica reducción de ruido y filtrado paso alto
def apply_denoise_and_highpass(
    audio_dataset: list[dict],
    denoise_strength: float = 0.7,
    highpass_cutoff: int = 100
) -> list[dict]:
    
    def highpass_filter(y: np.ndarray, sr: int, cutoff: int) -> np.ndarray:
        b, a = butter(5, cutoff / (sr / 2), btype='high')
        return filtfilt(b, a, y)
    
    output_dataset = []
    
    for item in audio_dataset:

        y = item["y"]
        sr = item["sr"]

        try:
            # Reducción de ruido
            y_denoised = nr.reduce_noise(
                y=y,
                sr=sr,
                prop_decrease=denoise_strength
            )

            # Filtro paso alto
            y_filtered = highpass_filter(
                y_denoised,
                sr,
                cutoff=highpass_cutoff
            )

            output_dataset.append({
                "audio_id": item["audio_id"],
                "y": y_filtered,
                "sr": sr
            })

        except Exception as e:
            print(f"Error procesando {item['audio_id']}: {e}")

    return output_dataset


# Aplica normalización RMS al audio para homogeneizar el nivel de energía
# Se utiliza un valor objetivo fijo seleccionado tras el análisis previo
def apply_rms_normalization(
    audio_dataset: list[dict],
    target_rms: float = 0.1
) -> list[dict]:
    
    output_dataset = []
    
    for item in audio_dataset:
        
        y = item["y"]
        sr = item["sr"]
        
        # Cálculo RMS actual
        rms_current = np.sqrt(np.mean(y**2))
        
        # Evitar división por cero
        if rms_current > 0:
            gain = target_rms / rms_current
            y_norm = y * gain
        else:
            y_norm = y
        
        output_dataset.append({
            "audio_id": item["audio_id"],
            "y": y_norm,
            "sr": sr
        })
    
    return output_dataset


# Aplica VAD basado en aprendizaje profundo (Silero) para eliminar silencios internos
# Solo se aplica a audios largos según la flag definida previamente
def apply_silero_vad(
    audio_dataset: list[dict],
    metadata_dataset: list[dict]
) -> list[dict]:
    
    model = load_silero_vad()
    
    output_dataset = []
    
    for item, meta in zip(audio_dataset, metadata_dataset):
        
        y = item["y"]
        sr = item["sr"]
        
        # Aplicar VAD solo a audios largos
        if meta["quality_flag"] == "too_long":
            
            # Conversión a tensor
            audio_tensor = torch.from_numpy(y).float()
            
            # Asegurar mono
            if audio_tensor.ndim > 1:
                audio_tensor = torch.mean(audio_tensor, dim=0)
            
            # Normalización para estabilidad del modelo
            if audio_tensor.abs().max() > 1:
                audio_tensor = audio_tensor / audio_tensor.abs().max()
            
            # Detección de voz
            speech_timestamps = get_speech_timestamps(
                audio_tensor,
                model,
                sampling_rate=sr
            )
            
            segments = [
                y[ts["start"]:ts["end"]]
                for ts in speech_timestamps
            ]
            
            output_dataset.append({
                "audio_id": item["audio_id"],
                "segments": segments,
                "sr": sr,
                "vad_applied": True
            })
        
        else:
            
            output_dataset.append({
                "audio_id": item["audio_id"],
                "segments": [y],
                "sr": sr,
                "vad_applied": False
            })
    
    return output_dataset


# Reconstruye el audio final uniendo los segmentos generados por VAD
def reconstruct_audio_from_segments(
    vad_dataset: list[dict]
) -> list[dict]:
    
    output_dataset = []
    
    for item in vad_dataset:
        
        segments = item["segments"]
        sr = item["sr"]
        
        if item["vad_applied"]:
            y_final = np.concatenate(segments)
        else:
            y_final = segments[0]
        
        output_dataset.append({
            "audio_id": item["audio_id"],
            "y": y_final,
            "sr": sr
        })
    
    return output_dataset


# Guarda los audios finales en disco listos para ASR
def save_audio_dataset(
    audio_dataset: list[dict],
    output_dir: Path
) -> None:
    
    for item in audio_dataset:
        
        output_path = output_dir / f"{item['audio_id']}.wav"
        
        sf.write(output_path, item["y"], item["sr"])

